In [0]:
spark.conf.set(
"fs.azure.account.key.varunsaidatalake.dfs.core.windows.net",
"<STORAGE_ACCOUNT_KEY>"
)

In [0]:
display(
dbutils.fs.ls("abfss://datalake@varunsaidatalake.dfs.core.windows.net/bronze/")
)

In [0]:
df = spark.read.parquet(
"abfss://datalake@varunsaidatalake.dfs.core.windows.net/bronze/nyc_taxi_raw/"
)

display(df)

In [0]:
df.printSchema()
df.count()

In [0]:
from pyspark.sql.functions import col
df_clean = df.filter(
    (col("trip_distance") > 0) &
    (col("fare_amount") > 0) &
    (col("passenger_count") > 0)
)

In [0]:
df_clean = df_clean.dropDuplicates()

In [0]:
from pyspark.sql.functions import to_date

df_clean = df_clean.withColumn(
    "pickup_date",
    to_date("tpep_pickup_datetime")
)

In [0]:
display(df_clean.limit(10))

In [0]:
df_clean.write.format("delta") \
.mode("overwrite") \
.save("abfss://datalake@varunsaidatalake.dfs.core.windows.net/silver/nyc_taxi/")

In [0]:
display(
dbutils.fs.ls("abfss://datalake@varunsaidatalake.dfs.core.windows.net/silver/")
)

In [0]:
silver_df = spark.read.format("delta").load(
"abfss://datalake@varunsaidatalake.dfs.core.windows.net/silver/nyc_taxi/"
)

display(silver_df)

In [0]:
from pyspark.sql.functions import sum

daily_revenue = silver_df.groupBy("pickup_date") \
    .agg(
        sum("fare_amount").alias("total_revenue")
    )

display(daily_revenue)

In [0]:
from pyspark.sql.functions import count

daily_trips = silver_df.groupBy("pickup_date") \
    .agg(
        count("*").alias("total_trips")
    )

display(daily_trips)

In [0]:
from pyspark.sql.functions import sum, count

gold_df = silver_df.groupBy("pickup_date").agg(
    count("*").alias("total_trips"),
    sum("fare_amount").alias("total_revenue"),
    sum("tip_amount").alias("total_tips")
)

display(gold_df)

In [0]:
gold_df.write.format("delta") \
.mode("overwrite") \
.save("abfss://datalake@varunsaidatalake.dfs.core.windows.net/gold/taxi_daily_metrics/")

In [0]:
display(
dbutils.fs.ls("abfss://datalake@varunsaidatalake.dfs.core.windows.net/gold/")
)

In [0]:
gold_table = spark.read.format("delta").load(
"abfss://datalake@varunsaidatalake.dfs.core.windows.net/gold/taxi_daily_metrics/"
)

display(gold_table)